# Step 4 - Model Development

**Baseline: SARIMAX with exogenous regressors.**

Cleaned dataset only - no engineered features.

Metrics are MAPE and RMSE per the brief, with WAPE alongside because MAPE is
undefined on zero-consumption days (12.2% of rows). Selection is on the validation
split; the test period is not touched.

In [1]:
import warnings

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

from mig_cement.config import settings

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

TRAIN_END, VAL_END = "2024-06-30", "2024-09-30"
TARGET = "y"

## 1. Load the cleaned dataset

In [2]:
clean = pd.read_parquet(settings.interim_dir / "operations_clean.parquet")
clean["date"] = pd.to_datetime(clean["date"])
clean = clean.sort_values(["site_id", "date"]).reset_index(drop=True)

print("shape:", clean.shape)
print("sites:", clean.site_id.nunique(), "| dates:", clean.date.nunique())
print("range:", clean.date.min().date(), "->", clean.date.max().date())

shape: (32880, 22)
sites: 30 | dates: 1096
range: 2022-01-01 -> 2024-12-31


## 2. Handle NaNs

Rows with NaN are removed, scoped to the columns this model uses.

A blanket `dropna()` would be a trap: `cover_days` is NaN exactly where
`consumed_tonnes == 0`, so it silently deletes every zero-consumption day - the
rain-blocked pours and stockouts, which are the hard cases.

In [3]:
print("NaN counts:")
print(clean.isna().sum()[lambda s: s > 0].to_string())
print("\nblanket dropna would give:", clean.dropna().shape,
      f"({100*(1-len(clean.dropna())/len(clean)):.1f}% lost)")
print("  zero-y rows before:", int((clean[TARGET] == 0).sum()),
      "| after:", int((clean.dropna()[TARGET] == 0).sum()))

NaN counts:
cover_days    4003

blanket dropna would give: (28877, 22) (12.2% lost)
  zero-y rows before: 4003 | after: 0


In [4]:
EXOG = ["planned_pour_tonnes", "rain_mm", "avg_temp_c", "opening_inventory_tonnes"]

before = len(clean)
clean = clean.dropna(subset=[TARGET] + EXOG).reset_index(drop=True)
print(f"rows: {before:,} -> {len(clean):,}")
print(f"zero-y rows retained: {int((clean[TARGET] == 0).sum()):,} "
      f"({(clean[TARGET] == 0).mean():.1%})")

rows: 32,880 -> 32,880
zero-y rows retained: 4,003 (12.2%)


### Why these four regressors

Of the 22 columns in the cleaned panel, most cannot be used:

- **target-derived / leaky**: `consumed_tonnes`, `served_tonnes`,
  `closing_inventory_tonnes`, `cover_days`, `silo_utilisation`, `was_constrained`,
  `unmet_tonnes`, `induced_shortfall`
- **not knowable at forecast time**: `deliveries_tonnes`, `received_tonnes`,
  `rejected_delivery_tonnes`
- **constant within each site**: `silo_capacity`, `region`, `behavior` - models are
  fitted per site, so these have no within-series variance and are collinear with
  the intercept
- **keys**: `date`, `site_id`, `cement_type`

## 3. Train / validation / test split

Chronological. The test period is held back.

In [5]:
d = clean["date"]
train = clean[d <= TRAIN_END]
val = clean[(d > TRAIN_END) & (d <= VAL_END)]
test = clean[d > VAL_END]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:6s} {len(part):6,} rows  {part.date.min().date()} -> {part.date.max().date()}")

train  27,360 rows  2022-01-01 -> 2024-06-30
val     2,760 rows  2024-07-01 -> 2024-09-30
test    2,760 rows  2024-10-01 -> 2024-12-31


## 4. Model configuration

`d = 0` because the series are stationary. `seasonal_order = (0,0,0,0)` because
Step 3 tested weekly, monthly and annual seasonality per region against a shuffled
null and found none - seasonal terms would fit noise.

In [6]:
adf = pd.Series({s: adfuller(g[TARGET])[1] for s, g in clean.groupby("site_id")})
print(f"ADF p-values across {len(adf)} sites: max = {adf.max():.2e}")
print(f"sites rejecting a unit root at 1%: {(adf < 0.01).sum()} / {len(adf)}")
print("\n-> d = 0")

ADF p-values across 30 sites: max = 3.61e-17
sites rejecting a unit root at 1%: 30 / 30

-> d = 0


In [7]:
# order chosen by mean AIC across a sample of sites
GRID = [(1, 0, 0), (0, 0, 1), (1, 0, 1), (2, 0, 1), (2, 0, 2)]
aic = {}
for o in GRID:
    scores = []
    for site in sorted(train.site_id.unique())[:5]:
        g = train[train.site_id == site].set_index("date")
        try:
            scores.append(SARIMAX(g[TARGET], exog=g[EXOG], order=o,
                                  seasonal_order=(0, 0, 0, 0)).fit(disp=False).aic)
        except Exception:
            pass
    aic[str(o)] = np.mean(scores)

aic = pd.Series(aic).sort_values()
print(aic.round(1).to_string())
ORDER = (2, 0, 2)
print("\nselected:", ORDER)

(2, 0, 2)    4979.0
(2, 0, 1)    4984.3
(1, 0, 1)    4984.7
(1, 0, 0)    4985.9
(0, 0, 1)    4986.2

selected: (2, 0, 2)


## 5. Train the model

Fitted on one site first, in the plainest form.

In [8]:
SITE = "SITE_001"

y_train = train[train.site_id == SITE].set_index("date")[TARGET]
x_train = train[train.site_id == SITE].set_index("date")[EXOG]
y_val = val[val.site_id == SITE].set_index("date")[TARGET]
x_val = val[val.site_id == SITE].set_index("date")[EXOG]

print(f"{SITE}: train {y_train.shape[0]} rows, val {y_val.shape[0]} rows, "
      f"{x_train.shape[1]} exogenous regressors")

SITE_001: train 912 rows, val 92 rows, 4 exogenous regressors


In [9]:
model = SARIMAX(
    y_train,
    exog=x_train,
    order=ORDER,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=True,
)

results = model.fit(disp=False)
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                      y   No. Observations:                  912
Model:               SARIMAX(2, 0, 2)   Log Likelihood               -3488.121
Date:                Thu, 06 Aug 2026   AIC                           6994.243
Time:                        11:58:29   BIC                           7037.584
Sample:                    01-01-2022   HQIC                          7010.789
                         - 06-30-2024                                         
Covariance Type:                  opg                                         
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
planned_pour_tonnes          0.6497      0.017     37.581      0.000       0.616       0.684
rain_mm                     -0.7636      0.054    -14.242      0.000      -0.869      -0.659
avg_temp_c                   0.1877      0.047      3.964      0.000       0.095       0.281
opening_inventory_tonnes     0.2729      0.020     13.435      0.000       0.233       0.313
ar.L1                        0.5999      1.215      0.494      0.622      -1.782       2.982
ar.L2                        0.2454      1.007      0.244      0.808      -1.729       2.220
ma.L1                       -0.5355      1.210     -0.443      0.658      -2.906       1.835
ma.L2                       -0.2605      0.941     -0.277      0.782      -2.104       1.583
sigma2                     123.0785      7.080     17.384      0.000     109.202     136.955
===================================================================================
Ljung-Box (L1) (Q):                   0.03   Jarque-Bera (JB):                60.37
Prob(Q):                              0.86   Prob(JB):                         0.00
Heteroskedasticity (H):               1.03   Skew:                            -0.63
Prob(H) (two-sided):                  0.82   Kurtosis:                         2.96
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

`enforce_stationarity=True` is deliberate. With it set to `False`, an unstable
AR root produced forecasts that diverged across the 92-day validation window -
one site reached an RMSE of 1.96e20. The series are stationary, so the constraint
costs nothing.

## 6. Forecast and score this site

In [10]:
pred = results.get_forecast(steps=len(y_val), exog=x_val).predicted_mean
pred = pred.clip(lower=0)


def metrics(y_true, y_pred, y_train=None):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    err = y_true - y_pred
    nz = np.abs(y_true) > 1e-9
    out = {
        "MAPE": np.mean(np.abs(err[nz] / y_true[nz])) if nz.any() else np.nan,
        "RMSE": np.sqrt(np.mean(err ** 2)),
        "WAPE": np.abs(err).sum() / np.abs(y_true).sum(),
        "MAE": np.mean(np.abs(err)),
        "bias": np.mean(y_pred - y_true),
        "pct_zero_actual": float((~nz).mean()),
    }
    if y_train is not None:
        scale = np.abs(np.diff(np.asarray(y_train, float))).mean()
        out["MASE"] = np.mean(np.abs(err)) / scale if scale else np.nan
    return out


pd.Series(metrics(y_val, pred, y_train)).to_frame(SITE).round(4)

,SITE_001
MAPE,0.2595
RMSE,10.7549
WAPE,0.2880
MAE,8.3536
bias,1.3598
pct_zero_actual,0.1739
MASE,0.5216


## 7. Fit all 30 sites

Same call, looped. This is what produces the site-level forecasts the project needs.

In [11]:
models, preds = {}, []

for site, g_tr in train.groupby("site_id"):
    g_tr = g_tr.set_index("date")
    g_va = val[val.site_id == site].sort_values("date").set_index("date")

    try:
        res = SARIMAX(
            g_tr[TARGET],
            exog=g_tr[EXOG],
            order=ORDER,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=True,
        ).fit(disp=False)
    except Exception:
        res = None

    models[site] = res
    p = (np.full(len(g_va), np.nan) if res is None else
         np.clip(res.get_forecast(steps=len(g_va), exog=g_va[EXOG]).predicted_mean.values, 0, None))
    preds.append(pd.DataFrame({"date": g_va.index, "site_id": site,
                               "actual": g_va[TARGET].values, "pred": p}))

fc = pd.concat(preds, ignore_index=True).dropna(subset=["pred"])
converged = sum(m is not None for m in models.values())
print(f"sites: {len(models)} | converged: {converged} | failed: {len(models) - converged}")
print(f"{len(fc):,} forecast rows | pred {fc.pred.min():.1f}-{fc.pred.max():.1f} t"
      f" | actual {fc.actual.min():.1f}-{fc.actual.max():.1f} t")

sites: 30 | converged: 30 | failed: 0
2,760 forecast rows | pred 0.0-69.0 t | actual 0.0-69.9 t


## 8. Metrics against baselines

In [12]:
rows = [
    {**metrics(val[TARGET], val["planned_pour_tonnes"], train[TARGET]),
     "model": "baseline: planned_pour"},
    {**metrics(val[TARGET], np.full(len(val), train[TARGET].mean()), train[TARGET]),
     "model": "baseline: train mean"},
    {**metrics(val[TARGET], val[TARGET].shift(1).fillna(train[TARGET].mean()), train[TARGET]),
     "model": "baseline: naive lag-1"},
    {**metrics(fc.actual, fc.pred, train[TARGET]), "model": "SARIMAX (cleaned data)"},
]

results_tbl = pd.DataFrame(rows).set_index("model")[
    ["MAPE", "RMSE", "WAPE", "MAE", "bias", "MASE", "pct_zero_actual"]]
results_tbl.round(4)

,MAPE,RMSE,WAPE,MAE,bias,MASE,pct_zero_actual
model,,,,,,,
baseline: planned_pour,0.3372,14.4142,0.3189,7.4622,7.4622,0.4969,0.1199
baseline: train mean,0.7077,16.6324,0.6094,14.2586,0.3923,0.9494,0.1199
baseline: naive lag-1,0.5631,20.9490,0.6438,15.0640,0.0022,1.0030,0.1199
SARIMAX (cleaned data),0.2227,8.6136,0.2464,5.7654,-0.0304,0.3839,0.1199


In [13]:
ref = results_tbl.loc["baseline: planned_pour"]
pd.DataFrame({
    "MAPE vs planned_pour": results_tbl.MAPE / ref.MAPE - 1,
    "RMSE vs planned_pour": results_tbl.RMSE / ref.RMSE - 1,
}).map(lambda x: f"{x:+.1%}")

,MAPE vs planned_pour,RMSE vs planned_pour
model,,
baseline: planned_pour,+0.0%,+0.0%
baseline: train mean,+109.9%,+15.4%
baseline: naive lag-1,+67.0%,+45.3%
SARIMAX (cleaned data),-33.9%,-40.2%


## 9. Error breakdown

In [14]:
fc["abs_err"] = (fc.actual - fc.pred).abs()
by_site = pd.DataFrame({
    "mean_actual": fc.groupby("site_id").actual.mean(),
    "RMSE": fc.groupby("site_id").abs_err.apply(lambda s: np.sqrt((s ** 2).mean())),
    "WAPE": fc.groupby("site_id").abs_err.sum() / fc.groupby("site_id").actual.sum(),
}).sort_values("WAPE", ascending=False)

print(f"WAPE across sites: best {by_site.WAPE.min():.3f} | "
      f"median {by_site.WAPE.median():.3f} | worst {by_site.WAPE.max():.3f}")
by_site.round(3)

WAPE across sites: best 0.059 | median 0.262 | worst 0.381


,mean_actual,RMSE,WAPE
site_id,,,
SITE_017,28.124,12.528,0.381
SITE_016,25.381,12.341,0.359
SITE_025,30.811,11.830,0.332
SITE_008,28.317,10.809,0.312
SITE_030,29.132,10.485,0.309
SITE_021,28.786,10.246,0.303
SITE_010,30.057,10.633,0.302
SITE_022,29.537,10.678,0.301
SITE_006,25.742,10.876,0.294


In [15]:
zero = fc.actual == 0
pd.DataFrame({
    "zero-consumption days": {"n": int(zero.sum()), "mean actual": 0.0,
                              "mean predicted": fc.pred[zero].mean(),
                              "MAE": fc.abs_err[zero].mean()},
    "pouring days": {"n": int((~zero).sum()), "mean actual": fc.actual[~zero].mean(),
                     "mean predicted": fc.pred[~zero].mean(),
                     "MAE": fc.abs_err[~zero].mean()},
}).round(2)

,zero-consumption days,pouring days
n,331.0,2429.00
mean actual,0.0,26.59
mean predicted,7.2,25.57
MAE,7.2,5.57


## Notes

- Cleaned dataset only, no engineered features.
- Weather regressors use actual validation values, which flatters the result -
  rain and temperature are not knowable eight weeks ahead.
- The test split is untouched.